## ML_1049: PPO (Proximal Policy Optimization) Clipped Loss

**Difficulty**: Hard | **TorchCode**: #39

### Problem
Implement the PPO clipped surrogate objective. Compute the probability ratio r = π_new/π_old, form both unclipped and clipped objectives, and take the more conservative (min) to prevent large policy updates.

### Formula
$$r_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}, \quad \mathcal{L}^{\text{CLIP}} = -\mathbb{E}\!\left[\min\!\left(r_t A_t,\; \text{clip}(r_t, 1\!-\!\varepsilon, 1\!+\!\varepsilon)A_t\right)\right]$$

In [ ]:
import torch
from torch import Tensor

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    old_logps_detached = old_logps.detach()
    adv_detached = advantages.detach()
    ratios = torch.exp(new_logps - old_logps_detached)
    unclipped = ratios * adv_detached
    clipped = torch.clamp(ratios, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv_detached
    return -torch.min(unclipped, clipped).mean()

In [ ]:
import torch
from torch import Tensor

# --- Test 1: Basic shape & type ---
new_logps = torch.randn(16, requires_grad=True)
old_logps = torch.randn(16)
advantages = torch.randn(16)
loss = ppo_loss(new_logps, old_logps, advantages)
assert isinstance(loss, Tensor) and loss.dim() == 0

# --- Test 2: Numeric check vs expected ---
new_logps2 = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps2 = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages2 = torch.tensor([1.0, -1.0, 0.5, -0.5])
loss2 = ppo_loss(new_logps2, old_logps2, advantages2, clip_ratio=0.2)
assert torch.allclose(loss2, torch.tensor(-0.0488), atol=1e-4)

# --- Test 3: Gradients flow to new_logps only ---
new_logps3 = torch.randn(8, requires_grad=True)
old_logps3 = torch.randn(8, requires_grad=True)
advantages3 = torch.randn(8, requires_grad=True)
ppo_loss(new_logps3, old_logps3, advantages3).backward()
assert new_logps3.grad is not None
assert old_logps3.grad is None
assert advantages3.grad is None

print('All tests passed!')